In [1]:
import os
import numpy as np
import pandas as pd


os.chdir(r'D:\Work\景观格局指标对PM2.5影响')

In [2]:
targets = pd.read_excel("建模数据/因变量.xlsx", index_col=0, usecols=[0, 1])
targets

,2.5_spring
SID,
S1_3,54
S2_3,48
S3_3,66
S5_3,42
S6_3,39
S7_3,58
S8_3,38
S11_3,28
S12_3,50


In [3]:
features = pd.read_excel("建模数据/自变量/01-Spring.xlsx", index_col=0)
features

,PREC,T2m,SP,RH,WS10,Dproad_300,Dproad_600,Dproad_900,Dproad_1200,Dproad_2400,...,ILP1200,ILP2400,ILP4800,Dt_eco,DPop_300,DPop_600,DPop_900,DPop_1200,DPop_2400,DPop_4800
SID,,,,,,,,,,,,,,,,,,,,,
S1_3,6.212903,12.984705,1017.157812,81.044292,1.270939,0.001265,0.004886,0.004276,0.003283,0.002129,...,0.000000,0.000000,2.171891,361.632300,0.019169,0.016845,0.019158,0.019414,0.020521,0.011583
S2_3,6.280645,12.958002,1015.867734,81.091632,1.284600,0.003665,0.005060,0.003766,0.003040,0.002945,...,1.292960,0.803401,1.049734,537.144125,0.022517,0.021921,0.019917,0.018502,0.023245,0.016714
S3_3,6.367742,13.055841,1018.716406,81.228142,1.338889,0.003867,0.002079,0.001404,0.002006,0.003388,...,31.572690,13.616941,5.423209,1703.372833,0.052762,0.043114,0.032077,0.023952,0.013400,0.010296
S5_3,6.367742,13.045435,1018.661484,81.308019,1.326964,0.004046,0.002140,0.002409,0.002014,0.001435,...,2.321013,7.648153,11.134880,2683.988108,0.004553,0.005657,0.007263,0.008037,0.007419,0.007594
S6_3,6.548387,13.038751,1017.229922,81.423260,1.388808,0.000000,0.000683,0.001152,0.002268,0.002263,...,0.000000,0.965198,9.131623,5486.573978,0.009205,0.010116,0.010849,0.011473,0.007790,0.002967
S7_3,6.493548,13.061334,1018.043359,81.348051,1.385824,0.000000,0.000000,0.000000,0.000941,0.002060,...,23.834006,28.784985,22.742945,7849.329414,0.002952,0.002335,0.001952,0.002103,0.002421,0.002614
S8_3,6.135484,12.705194,1008.021484,81.000024,1.201916,0.003186,0.006046,0.004468,0.004243,0.003037,...,2.888627,5.175177,3.513949,3083.411209,0.001526,0.001420,0.001571,0.001918,0.003002,0.002717
S11_3,6.364516,13.057428,1019.409531,81.367492,1.313293,0.003883,0.002144,0.002812,0.002862,0.001794,...,9.063763,6.784877,8.813477,527.706011,0.003108,0.003631,0.003809,0.003777,0.003236,0.003464
S12_3,6.216129,12.875787,1013.265469,81.078725,1.260117,0.002062,0.002932,0.004135,0.003471,0.002467,...,0.000000,0.000000,0.648915,901.066982,0.002692,0.003118,0.003076,0.002730,0.002438,0.004629


In [4]:
features_scaled = features.apply(lambda x: (x - x.min()) / (x.max() - x.min()))

In [5]:
def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

In [6]:
def calcMrer(y_true, y_pred):
    """
    计算MRER
    :param y_true: 实际值数组
    :param y_pred: 预测值数组
    :return: MRER值
    """
    relative_errors = np.abs((y_pred - y_true) / y_true)
    return np.mean(relative_errors)

In [88]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
        features_scaled[['ELP1200', 'SP', 'ILP1200']].values, targets.values.reshape(-1), train_size=0.8, test_size=0.2, random_state=1) 

In [89]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()

# 训练模型
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)

In [90]:
n = X_test.shape[0]
p =  X_test.shape[1]-1
print('系数:', model.coef_)
print('截距:', model.intercept_)
mse = mean_squared_error(y_test, y_pred)
print('标准估计误差see:', np.sqrt(mse * n / (n - p - 1)))
print('均方误差(MSE):', mse)
print('均方根误差(RMSE):', np.sqrt(mse))
r2 = r2_score(y_test, y_pred)
print("R2分数:", r2)
print('调整R2分数:', adjusted_r2(r2, n, p))
mrer = calcMrer(y_test, y_pred)
print("MRER: ", mrer)

系数: [-11.76008407  12.3447971    6.07319445]
截距: 35.083549484341155
标准估计误差see: 8.336039371593413
均方误差(MSE): 52.117164303566625
均方根误差(RMSE): 7.219221862747164
R2分数: 0.6688637636907169
调整R2分数: 0.5952779333997651
MRER:  0.15436043622794246


In [94]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
        features[['ELP1200', 'SP', 'ILP1200']].values, targets.values.reshape(-1), train_size=0.8, test_size=0.2, random_state=1) 

In [57]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

# 定义参数网格
param_grid = {
    # 'n_estimators': list(range(10, 200, 5)),
    # 'max_depth': list(range(1, 50, 2))
    'max_features': list(range(1, 50, 2))
}

# 创建网格搜索对象
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(
        n_estimators=75,
        max_depth=5,
        min_samples_leaf=2,
        min_samples_split=2,
        random_state=1),
    param_grid=param_grid,
    cv=5,
    n_jobs=24,
    verbose=2
)

# 执行网格搜索
grid_search.fit(X_train, y_train)

# 输出最佳参数
print("最佳参数:", grid_search.best_params_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
最佳参数: {'max_features': 3}


In [95]:
from sklearn.ensemble import RandomForestRegressor

rf_regressor = RandomForestRegressor(n_estimators=75,
                                     max_depth=5,
                                     min_samples_leaf=2,
                                     min_samples_split=2,
                                     max_features=3,
                                     random_state=1)

# 训练模型
rf_regressor.fit(X_train, y_train)

# 预测
y_pred = rf_regressor.predict(X_test)

In [96]:
n = X_test.shape[0]
p =  X_test.shape[1]-1
mse = mean_squared_error(y_test, y_pred)
print('标准估计误差see:', np.sqrt(mse * n / (n - p - 1)))
print('均方误差(MSE):', mse)
print('均方根误差(RMSE):', np.sqrt(mse))
r2 = r2_score(y_test, y_pred)
print("R2分数:", r2)
print('调整R2分数:', adjusted_r2(r2, n, p))
mrer = calcMrer(y_test, y_pred)
print("MRER: ", mrer)

标准估计误差see: 10.425444685394783
均方误差(MSE): 81.51742266616974
均方根误差(RMSE): 9.028699943301346
R2分数: 0.4820636752590699
调整R2分数: 0.36696671420552984
MRER:  0.19911201185347138
